# LeetCode #182 — Duplicate Emails (Tìm email xuất hiện trùng)

**Vị trí 42 trong danh sách Easy; bài SQL, học từ nền tảng.**  
Đề gốc: https://leetcode.com/problems/duplicate-emails/

> **Cách dùng notebook:** Đây là bài **Database**, LeetCode yêu cầu nộp **câu truy vấn SQL**, không phải `class Solution` Python. Ta chạy SQL thật bằng `sqlite3` (thư viện có sẵn trong Python) ngay trên Jupyter/Colab. Các đoạn Python dùng để dựng bảng, in kết quả, kiểm thử và hiểu ý tưởng.

**Sau bài này, bạn có thể tự làm được:**
1. Hiểu table/row/column/primary key; phân biệt email trùng với ID trùng.
2. Hiểu `SELECT`, `FROM`, `GROUP BY`, `COUNT(*)`, `HAVING` và `AS` từ ví dụ đơn giản.
3. Tự suy ra câu truy vấn tìm **mỗi email có từ 2 bản ghi trở lên**, chỉ in **một lần**.
4. Trace từng nhóm email, tự phân biệt `WHERE` với `HAVING`.
5. Viết lời giải SQL nộp chuẩn, thử edge cases; hiểu cách khác bằng `SELF JOIN`.
6. Làm 6 bài tập tăng dần, có TODO, 3 mức gợi ý và đáp án **cuối notebook**.

**Nhịp học:** đọc giải thích → dự đoán output → chạy code → đối chiếu. Không cần MySQL, pandas hay internet.

## 1. Đề bài thực sự hỏi điều gì?

Bảng **`Person`** có hai cột:

| id | email |
|---:|---|
| 1 | a@b.com |
| 2 | c@d.com |
| 3 | a@b.com |

- `id` là **mã của dòng/người**; là *primary key* nên không được lặp.
- `email` là **địa chỉ email**, có thể xuất hiện ở nhiều dòng.
- Bài yêu cầu **tìm tất cả email lặp lại ít nhất 2 lần**, không hỏi những người nào có email đó, cũng không yêu cầu xóa dòng trùng.

Đáp án ví dụ là một bảng có đúng **một cột tên `Email`** và một dòng `a@b.com`. Không được in `a@b.com` hai lần. Thứ tự các dòng kết quả không quan trọng.

**Điều kiện chính thức:** `email` chắc chắn khác `NULL` và không chứa chữ in hoa. Vì vậy ta **không cần tự thêm** thao tác chuẩn hóa chữ hoa hoặc xử lý email trống để giải đúng phạm vi đề.

In [1]:
person_rows = [(1, 'a@b.com'), (2, 'c@d.com'), (3, 'a@b.com')]
print('Số dòng ban đầu:', len(person_rows))
for person_id, email in person_rows:
    print('id =', person_id, '| email =', email)
print('Email trùng cần tìm: a@b.com')

Số dòng ban đầu: 3
id = 1 | email = a@b.com
id = 2 | email = c@d.com
id = 3 | email = a@b.com
Email trùng cần tìm: a@b.com


## 2. SQL là gì? Bảng khác danh sách Python ở đâu?

**SQL (Structured Query Language)** là ngôn ngữ truy vấn bảng dữ liệu. Thay vì tự duyệt từng dòng bằng `for`, ta mô tả: lấy dữ liệu từ bảng nào, gom nhóm theo cái gì, lọc nhóm theo điều kiện nào.

- **Table (bảng):** tương tự trang tính Excel.
- **Row (dòng/bản ghi):** một người có id và email.
- **Column (cột):** thuộc tính `id` hoặc `email`.
- **Primary key (khóa chính):** ở đây là `id`, phải duy nhất; **không có nghĩa `email` duy nhất**.
- `SELECT` chọn cột cần xem; `FROM` chỉ ra bảng.

`SELECT email FROM Person;` sẽ xuất tất cả email theo từng dòng, **kể cả dòng trùng**. Dấu `;` thường dùng để kết thúc một câu SQL. Học thử trên database thật ngay sau đây.

## 3. Dùng SQLite trong Python để học SQL mà không cần cài server

`sqlite3` là thư viện đi kèm Python. `sqlite3.connect(':memory:')` tạo **cơ sở dữ liệu tạm trong RAM**. Mọi câu SQL ta viết đều được SQLite thực thi thật, không phải Python giả lập SQL.

- `conn.execute(sql)` gửi câu lệnh SQL tới cơ sở dữ liệu.
- `conn.executemany(sql, rows)` chạy câu lệnh nhiều lần với từng bộ giá trị.
- `conn.commit()` xác nhận dữ liệu đã thêm.
- `fetchall()` lấy tất cả dòng đầu ra.

SQL dùng ở đây tương thích với cú pháp lời giải MySQL cơ bản trên LeetCode. **Chỉ copy câu `SELECT` cuối bài khi nộp**, không copy phần `import sqlite3`.

In [2]:
import sqlite3
db182 = sqlite3.connect(':memory:')
print('SQLite đã sẵn sàng:', sqlite3.sqlite_version)

SQLite đã sẵn sàng: 3.46.1


## 4. Tạo đúng bảng `Person(id, email)`

Câu `CREATE TABLE Person (...)` giống như xây một bảng trống trước khi ghi nội dung.

- `INTEGER PRIMARY KEY`: cột số nguyên `id` làm khóa chính.
- `TEXT NOT NULL`: một chuỗi và không nhận giá trị `NULL`. Trong mô tả LeetCode, `email` dùng `varchar`, còn SQLite minh họa bằng `TEXT`.

**Lưu ý:** `NOT NULL` chỉ có nghĩa không được thiếu; **không** bảo đảm email không lặp. Nếu ta khai báo `email UNIQUE` thì lại vô tình **cấm tình huống bài toán đang muốn phát hiện**.

In [3]:
db182.execute('CREATE TABLE Person (id INTEGER PRIMARY KEY, email TEXT NOT NULL);')
db182.executemany('INSERT INTO Person (id, email) VALUES (?, ?);', person_rows)
db182.commit()
print('Đã tạo bảng và thêm', len(person_rows), 'dòng')

Đã tạo bảng và thêm 3 dòng


## 5. Viết hàm nhỏ để quan sát kết quả SQL

Khi chạy `db182.execute(query)`, ta nhận một **cursor** — đối tượng giúp đọc kết quả. `cursor.description` chứa tên các cột; `fetchall()` đọc từng dòng dưới dạng tuple Python.

Ví dụ SQL `SELECT email FROM Person;` trả ra 3 dòng, dù có 2 email giống nhau. Hàm `show_182` giúp ta **nhìn** SQL, không làm thay SQL. `ORDER BY` nếu dùng trong phòng lab chỉ để có thứ tự dễ quan sát; đề cho phép thứ tự tùy ý.

In [4]:
def show_182(query, db=db182):
    cur = db.execute(query)
    columns = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print('Cột:', columns)
    for row in rows:
        print('  ', row)
    print('Tổng số dòng:', len(rows))
    return rows

show_182('SELECT id, email FROM Person ORDER BY id;')

Cột: ['id', 'email']
   (1, 'a@b.com')
   (2, 'c@d.com')
   (3, 'a@b.com')
Tổng số dòng: 3


[(1, 'a@b.com'), (2, 'c@d.com'), (3, 'a@b.com')]

## 6. `SELECT` và `FROM`: chọn đúng cột, đúng bảng

Đọc `SELECT email FROM Person;` thành câu: “Từ bảng Person, lấy cột email”. `SELECT *` nghĩa là lấy toàn bộ cột (kể cả `id`).

**Hai lỗi nên tránh:**
1. Dùng `SELECT id` trong lời giải: id có tính duy nhất nên không thể là câu trả lời email trùng.
2. Dùng `SELECT *`: đề chỉ muốn **một cột `Email`**, không muốn thêm id.

Ta chỉ đang học cú pháp; **chưa tìm được email trùng** ở bước này.

In [5]:
show_182('SELECT email FROM Person ORDER BY id;')
show_182('SELECT * FROM Person ORDER BY id;')

Cột: ['email']
   ('a@b.com',)
   ('c@d.com',)
   ('a@b.com',)
Tổng số dòng: 3
Cột: ['id', 'email']
   (1, 'a@b.com')
   (2, 'c@d.com')
   (3, 'a@b.com')
Tổng số dòng: 3


[(1, 'a@b.com'), (2, 'c@d.com'), (3, 'a@b.com')]

## 7. `WHERE`: lọc từng dòng *trước khi gom nhóm*

Câu `SELECT email FROM Person WHERE id > 1;` giữ lại các dòng có id bằng 2 hoặc 3. Trong SQL, so sánh bằng dùng **`=`**, còn trong Python ta thường dùng `==`.

`WHERE` phù hợp khi điều kiện áp dụng được ngay cho **một dòng**: `id > 1`, `email = 'a@b.com'`...

Nhưng câu hỏi bài này là **“email nào xuất hiện trên ít nhất hai dòng?”**. Một dòng đơn lẻ không chứa thông tin **số lần email đó xuất hiện trên toàn bảng**. Đây là lý do ta cần học `GROUP BY` và `COUNT`.

In [6]:
show_182('SELECT id, email FROM Person WHERE id > 1 ORDER BY id;')
show_182("SELECT id, email FROM Person WHERE email = 'a@b.com' ORDER BY id;")

Cột: ['id', 'email']
   (2, 'c@d.com')
   (3, 'a@b.com')
Tổng số dòng: 2
Cột: ['id', 'email']
   (1, 'a@b.com')
   (3, 'a@b.com')
Tổng số dòng: 2


[(1, 'a@b.com'), (3, 'a@b.com')]

## 8. Cách làm trực giác bằng Python: đếm từng email

Khi mới gặp bài, ta có thể suy nghĩ: lấy từng email, đi qua **tất cả các dòng** để đếm số lần nó xuất hiện. Nếu đếm được từ 2 trở lên, đưa email đó vào kết quả.

Nhưng nếu kiểm tra từng dòng, cùng một email có thể bị đưa vào kết quả nhiều lần. Ta dùng một `set` (tập hợp) để chỉ giữ **giá trị khác nhau**.

- `set()` tạo tập rỗng; `seen_emails.add(email)` thêm vào tập.
- `sum(condition for ...)` đếm điều kiện True vì `True` được tính là 1.
- Đây là cách Python để **hiểu bài toán**, không phải code SQL nộp LeetCode.

Gọi `n` là số dòng: nếu kiểm tra từng email trong n dòng, baseline có thể tốn `O(n²)` thời gian.

In [7]:
def duplicate_emails_baseline(rows):
    result = set()
    for _, candidate in rows:
        count = 0
        for _, email in rows:
            if email == candidate:
                count += 1
        if count > 1:
            result.add(candidate)
    return sorted(result)

print('Ví dụ:', duplicate_emails_baseline(person_rows))
print('Không có email trùng:', duplicate_emails_baseline([(1, 'a@b.com'), (2, 'b@b.com')]))

Ví dụ: ['a@b.com']
Không có email trùng: []


## 9. Nút thắt của baseline: đếm đi đếm lại cùng một giá trị

Với `[(1, 'a'), (2, 'b'), (3, 'a')]`, baseline khi xét dòng 1 đếm `'a'` hai lần; tới dòng 3 lại đếm `'a'` thêm lần nữa — **cùng một công việc bị lặp**.

Ta mong muốn cơ sở dữ liệu **gom các dòng có cùng email thành một nhóm**. Khi đó, mỗi nhóm có một giá trị email và một số đếm riêng. Đây chính là ý tưởng `GROUP BY email`.

Đừng học thuộc ngay `GROUP BY email HAVING COUNT(*) > 1`. Hãy xem nhóm trông như thế nào và số đếm xuất hiện ở đâu.

## 10. `GROUP BY`: gom những email giống nhau lại

`GROUP BY email` yêu cầu SQL chia các dòng Person thành **các nhóm có cùng giá trị email**.

Với dữ liệu ví dụ:

| Nhóm email | Các id thuộc nhóm |
|---|---|
| `a@b.com` | 1, 3 |
| `c@d.com` | 2 |

Cần nhớ: `GROUP BY email` không tự xóa dòng khỏi bảng gốc. Nó chỉ **tổ chức các dòng thành nhóm để tính toán** trong truy vấn. Khi `SELECT email` kèm `GROUP BY email`, ta thấy mỗi email một dòng kết quả.

In [8]:
show_182('SELECT email FROM Person GROUP BY email ORDER BY email;')

Cột: ['email']
   ('a@b.com',)
   ('c@d.com',)
Tổng số dòng: 2


[('a@b.com',), ('c@d.com',)]

## 11. `COUNT(*)`: đếm số dòng của từng nhóm

`COUNT(*)` là một **aggregate function (hàm tổng hợp)**: đếm bao nhiêu dòng trong nhóm hiện tại. Dấu `*` ở đây **không có nghĩa chọn mọi cột**; nó chỉ bảo `COUNT` đếm các dòng.

```sql
SELECT email, COUNT(*) AS so_lan
FROM Person
GROUP BY email;
```

- `COUNT(*)` được tính **cho từng nhóm**, chứ không phải tổng mọi dòng (vì có GROUP BY).
- `AS so_lan` đặt **tên hiển thị** dễ hiểu cho cột đếm; nó không sửa tên cột trong bảng gốc.
- Kết quả cần đạt: a@b.com → 2; c@d.com → 1.

In [9]:
group_counts = show_182('''SELECT email, COUNT(*) AS so_lan FROM Person GROUP BY email ORDER BY email;''')
assert group_counts == [('a@b.com', 2), ('c@d.com', 1)]

Cột: ['email', 'so_lan']
   ('a@b.com', 2)
   ('c@d.com', 1)
Tổng số dòng: 2


## 12. Phân biệt `COUNT(*)`, `COUNT(email)` và `COUNT(DISTINCT email)`

- `COUNT(*)`: đếm **số dòng**.
- `COUNT(email)`: đếm số dòng có `email` **khác NULL**. Bài #182 đảm bảo `email` không NULL → hai con số này bằng nhau.
- `COUNT(DISTINCT email)`: đếm **số giá trị email khác nhau**; dùng trong nhóm `GROUP BY email` sẽ chỉ bằng 1, không giúp phát hiện lặp.

Trước khi chạy, hãy dự đoán: toàn bảng 3 dòng, có bao nhiêu email phân biệt? Vì sao hai kết quả khác nhau?

In [10]:
show_182('SELECT COUNT(*) AS tong_dong, COUNT(email) AS co_email, COUNT(DISTINCT email) AS email_khac_nhau FROM Person;')
show_182('SELECT email, COUNT(DISTINCT email) AS dem_sai_muc_dich FROM Person GROUP BY email ORDER BY email;')

Cột: ['tong_dong', 'co_email', 'email_khac_nhau']
   (3, 3, 2)
Tổng số dòng: 1
Cột: ['email', 'dem_sai_muc_dich']
   ('a@b.com', 1)
   ('c@d.com', 1)
Tổng số dòng: 2


[('a@b.com', 1), ('c@d.com', 1)]

## 13. Bước đột phá: `HAVING` lọc **nhóm** sau khi đếm

Bây giờ mỗi email có một số đếm. Ta cần chỉ giữ nhóm có **`COUNT(*) > 1`**.

```sql
SELECT email, COUNT(*) AS so_lan
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
```

**`WHERE` và `HAVING` khác nhau ở thời điểm/đối tượng lọc:**

- `WHERE`: điều kiện trên dòng dữ liệu, **trước khi tạo nhóm**.
- `HAVING`: điều kiện trên nhóm đã hình thành, **sau khi tính số đếm**.

Ví dụ: email a có 2 dòng → giữ; email c có 1 dòng → bỏ. `COUNT(*) > 1` là “ít nhất 2 lần”, không phải “hơn 2 lần”.

In [11]:
show_182('SELECT email, COUNT(*) AS so_lan FROM Person GROUP BY email HAVING COUNT(*) > 1;')

Cột: ['email', 'so_lan']
   ('a@b.com', 2)
Tổng số dòng: 1


[('a@b.com', 2)]

## 14. Trace thủ công đầy đủ: nhìn dữ liệu → nhóm → lọc

Giả sử bảng có các email: `a`, `c`, `a`, `b`, `b`, `b`, `d`.

| Bước | Nhóm email | Dòng thuộc nhóm | `COUNT(*)` | `> 1`? | Kết quả |
|---:|---|---|---:|---|---|
| 1 | a | 1, 3 | 2 | Đúng | lấy `a` |
| 2 | b | 4, 5, 6 | 3 | Đúng | lấy `b` |
| 3 | c | 2 | 1 | Sai | bỏ `c` |
| 4 | d | 7 | 1 | Sai | bỏ `d` |

**Điểm quan trọng:** `GROUP BY` tạo đúng một kết quả cho mỗi email; `HAVING` quyết định giữ/bỏ **cả nhóm**, không giữ từng dòng email nguyên bản.

In [12]:
trace_emails = ['a', 'c', 'a', 'b', 'b', 'b', 'd']
for email in sorted(set(trace_emails)):
    indices = [i + 1 for i, value in enumerate(trace_emails) if value == email]
    count = len(indices)
    print(f'Nhóm {email}: dòng {indices}, COUNT={count}, giữ? {count > 1}')

Nhóm a: dòng [1, 3], COUNT=2, giữ? True
Nhóm b: dòng [4, 5, 6], COUNT=3, giữ? True
Nhóm c: dòng [2], COUNT=1, giữ? False
Nhóm d: dòng [7], COUNT=1, giữ? False


## 15. Đọc một câu SQL từ dưới lên để không bị rối

Câu nộp chuẩn chỉ có 4 ý:

```sql
SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
```

**Thứ tự tư duy:**
1. `FROM Person`: lấy dữ liệu ở bảng Person.
2. `GROUP BY email`: gom cùng email.
3. `HAVING COUNT(*) > 1`: giữ nhóm xuất hiện nhiều lần.
4. `SELECT email AS Email`: xuất **chính email đó**, đổi tên cột thành `Email`.

`Email` viết hoa chữ E là tên cột theo yêu cầu đầu ra. **Không trả ra cột `COUNT(*)`**: đếm chỉ là thao tác trung gian để quyết định nhóm nào xuất hiện.

In [13]:
sql182 = '''SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;'''
actual182 = show_182(sql182)
assert actual182 == [('a@b.com',)]
print('PASS — cột tên Email, kết quả đúng ví dụ LeetCode')

Cột: ['Email']
   ('a@b.com',)
Tổng số dòng: 1
PASS — cột tên Email, kết quả đúng ví dụ LeetCode


## 16. Code SQL nộp LeetCode (đúng định dạng)

Đây là toàn bộ câu lệnh cần dán vào ô **MySQL / SQL** trong LeetCode. Không có `import`, `conn.execute`, không có `class Solution`.

```sql
SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
```

Bảng kết quả cho phép **bất kỳ thứ tự dòng nào**, vì vậy câu nộp không bắt buộc `ORDER BY`. Nếu có nhiều email trùng, `GROUP BY` đảm bảo mỗi email xuất **một lần**.

Trong cell sau, ta xác nhận SQLite trả tên cột chuẩn `Email` (không phải `email` hay `so_lan`).

In [14]:
cursor182 = db182.execute(sql182)
print('Tên cột kết quả:', [d[0] for d in cursor182.description])
print('Dòng kết quả:', cursor182.fetchall())
assert cursor182.description[0][0] == 'Email'

Tên cột kết quả: ['Email']
Dòng kết quả: [('a@b.com',)]


## 17. Test nhiều dữ liệu mà không làm thay đổi ví dụ ban đầu

Khi học SQL, các lệnh `INSERT` thường làm bảng **thay đổi vĩnh viễn trong kết nối hiện tại**, dễ khiến các test sau không còn giống ban đầu. Để tránh phụ thuộc trạng thái ngầm, ta viết `run_182(rows, query)` tạo **database mới trong RAM** cho mỗi ca thử, rồi đóng lại.

**Phân biệt hai tầng:**
- `run_182`: công cụ kiểm thử viết bằng Python.
- `sql182`: thuật toán SQL thực sự được chấm.

Đề cho phép thứ tự email bất kỳ → khi so sánh ta dùng `set` hoặc `sorted`, **không** khẳng định SQL luôn trả theo ABC nếu không có `ORDER BY`.

In [15]:
def run_182(rows, query=sql182):
    connection = sqlite3.connect(':memory:')
    try:
        connection.execute('CREATE TABLE Person (id INTEGER PRIMARY KEY, email TEXT NOT NULL);')
        connection.executemany('INSERT INTO Person VALUES (?, ?);', rows)
        cur = connection.execute(query)
        return [col[0] for col in cur.description], cur.fetchall()
    finally:
        connection.close()

print(run_182([(10,'a'),(20,'b'),(30,'a'),(40,'b'),(50,'b')]))

(['Email'], [('a',), ('b',)])


## 18. Kiểm thử ví dụ và các trường hợp dễ sai

Những trường hợp nên chủ động kiểm tra:

1. **Bảng rỗng:** không có email trùng → kết quả rỗng.
2. **Chỉ 1 dòng:** không thể lặp → rỗng.
3. **Tất cả khác nhau:** không có nhóm đủ 2 → rỗng.
4. **Xuất hiện đúng 2 lần:** phải được chọn; đây là lý do dùng `> 1`, không phải `> 2`.
5. **3 lần hoặc hơn:** vẫn chỉ xuất **một dòng**.
6. **Nhiều nhóm trùng:** phải xuất tất cả nhóm; `id` có thể không liên tiếp.

Ta dùng `assert` để báo lỗi ngay nếu `actual` khác `expected`.

In [16]:
cases182 = [
    ('rỗng', [], set()),
    ('một dòng', [(1,'a')], set()),
    ('toàn khác nhau', [(1,'a'),(2,'b'),(3,'c')], set()),
    ('đúng hai lần', [(1,'a'),(4,'a')], {'a'}),
    ('ba lần', [(1,'a'),(2,'a'),(3,'a')], {'a'}),
    ('hai nhóm', [(10,'x'),(7,'y'),(20,'y'),(31,'x'),(40,'x')], {'x','y'}),
]
for name, rows, expected in cases182:
    columns, actual = run_182(rows)
    assert columns == ['Email']
    assert {row[0] for row in actual} == expected, (name, actual, expected)
    assert len(actual) == len(expected), (name, 'trùng trong output')
    print(name, 'PASS')

rỗng PASS
một dòng PASS
toàn khác nhau PASS
đúng hai lần PASS
ba lần PASS
hai nhóm PASS


## 19. Vì sao `WHERE COUNT(*) > 1` là lỗi?

Cú pháp dễ viết nhầm:

```sql
-- SAI: WHERE không dùng để lọc COUNT(*) của một nhóm
SELECT email FROM Person WHERE COUNT(*) > 1 GROUP BY email;
```

Lý do: `COUNT(*)` cần các dòng **được tập hợp thành nhóm** trước; `WHERE` diễn ra trước bước đó. Trong SQLite đây thường là lỗi *misuse of aggregate function*.

Sửa bằng:

```sql
SELECT email AS Email FROM Person GROUP BY email HAVING COUNT(*) > 1;
```

Không nên chạy lệnh sai như code mẫu chính; cell dưới đây bắt lỗi có chủ ý **để quan sát** mà vẫn không khiến notebook dừng.

In [17]:
wrong_query_182 = 'SELECT email FROM Person WHERE COUNT(*) > 1 GROUP BY email;'
try:
    db182.execute(wrong_query_182).fetchall()
except sqlite3.OperationalError as error:
    print('Lỗi có chủ ý của WHERE COUNT(*):', error)
print('Cách đúng:', db182.execute(sql182).fetchall())

Lỗi có chủ ý của WHERE COUNT(*): misuse of aggregate: COUNT()
Cách đúng: [('a@b.com',)]


## 20. Vì sao `SELECT DISTINCT email` chưa đủ?

`DISTINCT` loại bỏ các giá trị email **giống nhau trong kết quả**, bất kể email xuất hiện 1 hay 10 lần.

Với `a, a, b, c`, `SELECT DISTINCT email` tạo `a, b, c` — nhưng bài chỉ cần `a`.

`GROUP BY ... HAVING COUNT(*) > 1` làm **hai việc**: nhóm giá trị và kiểm tra **tần suất**. `DISTINCT` tự nó không biết tần suất. Đây là một kỹ năng quan trọng khi đọc yêu cầu: **“unique values” khác “values occurring at least twice”**.

In [18]:
print('DISTINCT:')
show_182('SELECT DISTINCT email FROM Person ORDER BY email;')
print('Đúng yêu cầu #182:')
show_182(sql182)

DISTINCT:
Cột: ['email']
   ('a@b.com',)
   ('c@d.com',)
Tổng số dòng: 2
Đúng yêu cầu #182:
Cột: ['Email']
   ('a@b.com',)
Tổng số dòng: 1


[('a@b.com',)]

## 21. Một cách khác: duyệt bảng `Person` hai lần bằng `SELF JOIN`

**Self join** = dùng **cùng một bảng ở hai vai trò khác nhau**. Ta lấy `Person p1` và `Person p2`, kiểm tra:

- `p1.email = p2.email`: hai dòng có cùng email.
- `p1.id <> p2.id`: phải là **hai dòng khác nhau**, không phải một dòng tự so với chính nó.
- `DISTINCT`: một email có thể ghép thành nhiều cặp, chỉ xuất một lần.

```sql
SELECT DISTINCT p1.email AS Email
FROM Person AS p1
JOIN Person AS p2
    ON p1.email = p2.email AND p1.id <> p2.id;
```

Đây là một **lời giải hợp lệ**, nhưng `GROUP BY + HAVING` thường dễ đọc hơn cho bài **đếm tần suất**. Từ khóa `<>` trong SQL nghĩa là **khác**, tương đương `!=` của Python.

In [19]:
sql182_join = '''SELECT DISTINCT p1.email AS Email
FROM Person AS p1
JOIN Person AS p2
  ON p1.email = p2.email AND p1.id <> p2.id;'''
show_182(sql182_join)
assert set(db182.execute(sql182_join).fetchall()) == set(db182.execute(sql182).fetchall())

Cột: ['Email']
   ('a@b.com',)
Tổng số dòng: 1


## 22. Trace self join: vì sao phải loại `p1.id = p2.id`?

Với email `a@b.com` ở id 1 và 3:

| p1.id | p2.id | Cùng email? | Khác id? | Được ghép? |
|---:|---:|---|---|---|
| 1 | 1 | Đúng | **Sai** | Không |
| 1 | 3 | Đúng | Đúng | Có |
| 3 | 1 | Đúng | Đúng | Có |
| 3 | 3 | Đúng | **Sai** | Không |

Nếu thiếu `p1.id <> p2.id`, **mọi email**, kể cả chỉ xuất hiện một lần, đều ghép được với chính nó → sai. Nếu thiếu `DISTINCT`, `a@b.com` sẽ có **hai dòng** kết quả. Hãy nhìn thử kết quả ghép thô.

In [20]:
show_182('''SELECT p1.id, p2.id, p1.email FROM Person p1 JOIN Person p2 ON p1.email = p2.email AND p1.id <> p2.id ORDER BY p1.id,p2.id;''')

Cột: ['id', 'id', 'email']
   (1, 3, 'a@b.com')
   (3, 1, 'a@b.com')
Tổng số dòng: 2


[(1, 3, 'a@b.com'), (3, 1, 'a@b.com')]

## 23. So sánh với `dict` Python — tần suất được lưu ở đâu?

Ta có thể mô phỏng ý tưởng nhóm bằng **dictionary (từ điển)** Python:

- `counts = {}`: ánh xạ `email → số lần gặp`.
- `counts.get(email, 0)`: đọc số đếm; nếu chưa có email trả 0.
- `counts[email] += 1`: cập nhật lần xuất hiện.

Đây là cách rất gần với trực giác của `GROUP BY + COUNT(*)`; nhưng ở SQL ta **giao cho hệ quản trị dữ liệu gom nhóm**, không tự dùng dict trong câu nộp. Ví dụ bên dưới có in trạng thái **trước/sau** từng lượt.

In [21]:
counts182 = {}
for i, (_, email) in enumerate(person_rows):
    before = counts182.get(email, 0)
    counts182[email] = before + 1
    print(f'Lượt {i}: {email:<8} | trước={before} -> sau={counts182[email]} | dict={counts182}')
print('Chọn nhóm đếm > 1:', [mail for mail,count in counts182.items() if count > 1])

Lượt 0: a@b.com  | trước=0 -> sau=1 | dict={'a@b.com': 1}
Lượt 1: c@d.com  | trước=0 -> sau=1 | dict={'a@b.com': 1, 'c@d.com': 1}
Lượt 2: a@b.com  | trước=1 -> sau=2 | dict={'a@b.com': 2, 'c@d.com': 1}
Chọn nhóm đếm > 1: ['a@b.com']


## 24. Vì sao lời giải đúng? Chứng minh bằng một nhóm email

Xét một email bất kỳ `e`:

1. Tất cả dòng có `email = e` được `GROUP BY email` gom **vào đúng một nhóm** vì chúng có cùng giá trị.
2. `COUNT(*)` của nhóm bằng đúng số dòng chứa email `e`.
3. `HAVING COUNT(*) > 1` giữ nhóm **khi và chỉ khi** `e` xuất hiện ít nhất hai lần.
4. `SELECT email AS Email` lấy **một dòng cho nhóm đó**.

Vì vậy ta **không bỏ sót** email nào lặp lại, **không trả nhầm** email chỉ xuất hiện một lần, và **không in nhiều lần** cho cùng email. Đây là lý do logic để tin lời giải, độc lập với các test.

## 25. Time/Space Complexity — đừng áp công thức Python máy móc lên SQL

Đặt `n` là **số dòng bảng Person**, `k` là **số email khác nhau**.

- **Brute Force Python** như mục 8: `O(n²)` thời gian; `O(k)` không gian để giữ kết quả (không tính input).
- **Đếm bằng dict Python**: khoảng `O(n)` thời gian **trung bình**, `O(k)` bộ nhớ phụ; do tra cứu hash trung bình nhanh. Không mặc định trường hợp xấu nhất cũng `O(n)`.
- **`GROUP BY + HAVING` trong SQL**: độ phức tạp *vật lý* phụ thuộc hệ quản trị, chỉ mục và kế hoạch thực thi. Nếu engine gom nhóm bằng hash thì thường gần `O(n)` thời gian dự kiến + bộ nhớ cho các nhóm; nếu phải sắp xếp thì có thể gần `O(n log n)` với bộ nhớ tùy triển khai. Không nên khẳng định SQL **luôn** `O(n)` hoặc `O(1)` bộ nhớ.
- Truy vấn `SELF JOIN` có thể tạo nhiều cặp, đặc biệt nếu email xuất hiện nhiều lần; không phải tối ưu mặc định cho bài đếm.

**Điều cần nhớ để phỏng vấn:** giải thích nhóm dữ liệu trước; sau đó nói độ phức tạp SQL là **phụ thuộc query plan**.

## 26. Xem `EXPLAIN QUERY PLAN` — SQL cũng có bước thực thi phía sau

Khi dùng SQLite, `EXPLAIN QUERY PLAN` cho ta biết một **tóm tắt kế hoạch chạy truy vấn**. Nó không làm thay đổi câu SQL và không phải phần cần nộp.

Với bảng minh họa nhỏ, SQLite có thể dùng **temporary B-tree** để gom nhóm. Đó là lựa chọn của một engine trên bộ dữ liệu nhất định, **không phải** kết luận rằng mọi MySQL/SQL đều thực thi giống hệt.

Hãy xem đầu ra và tìm các từ như `SCAN` (đọc bảng), `GROUP BY` hoặc `TEMP B-TREE`.

In [22]:
print('Kế hoạch SQLite minh họa:')
for entry in db182.execute('EXPLAIN QUERY PLAN ' + sql182):
    print(entry)

Kế hoạch SQLite minh họa:
(6, 0, 0, 'SCAN Person')
(8, 0, 0, 'USE TEMP B-TREE FOR GROUP BY')


## 27. Test đối chiếu baseline và SQL trên nhiều mẫu

Khi có hai cách giải dựa trên nguyên lý khác nhau, ta có thể **kiểm tra chéo** để tìm lỗi.

Ta tạo nhiều danh sách email ngắn với `random.Random(182)` — **seed cố định** giúp mỗi lần chạy tạo cùng dữ liệu. ID là `i+1`, luôn duy nhất. `sorted` dùng để so kết quả như **tập giá trị**, không phụ thuộc thứ tự hàng SQL.

Kiểm thử ngẫu nhiên giúp tăng tự tin, **không thay thế** chứng minh logic ở mục 24.

In [23]:
import random
rng182 = random.Random(182)
for trial in range(100):
    emails = [rng182.choice(['a','b','c','d','e']) for _ in range(rng182.randrange(0,20))]
    rows = [(i+1, em) for i, em in enumerate(emails)]
    expected = duplicate_emails_baseline(rows)
    actual = sorted(row[0] for row in run_182(rows)[1])
    assert actual == expected, (trial, rows, expected, actual)
print('PASS: 100 ca ngẫu nhiên so SQL với baseline Python')

PASS: 100 ca ngẫu nhiên so SQL với baseline Python


## 28. Những lỗi thường gặp (code sai → vì sao → sửa)

| Sai lầm | Vì sao sai? | Cách sửa |
|---|---|---|
| `SELECT DISTINCT email FROM Person` | Cả email xuất hiện 1 lần cũng được trả | Dùng `HAVING COUNT(*) > 1` |
| `WHERE COUNT(*) > 1` | Lọc trước khi gom nhóm | Chuyển điều kiện sang `HAVING` |
| `HAVING COUNT(*) > 2` | Bỏ email trùng **đúng 2 lần** | Dùng `> 1` hoặc `>= 2` |
| `SELECT id` | ID khóa chính không trùng | Chọn `email` |
| `SELECT email, COUNT(*)` khi submit | Thừa cột số đếm | `SELECT email AS Email` |
| Self join thiếu `p1.id <> p2.id` | Ghép một người với chính mình | Thêm điều kiện khác ID |
| Self join thiếu `DISTINCT` | Có thể xuất một email nhiều lần | Dùng `DISTINCT` |

**Mẹo phát hiện lỗi:** hãy tự tạo dữ liệu `a, a, b`, rồi dự đoán một câu sai sẽ trả ra những dòng nào.

## 29. Mẹo nhận dạng pattern: “đếm theo nhóm”

Dấu hiệu thường gặp của `GROUP BY + HAVING`:

- “Những giá trị nào **xuất hiện từ 2 lần trở lên**?”
- “Khách hàng nào có **trên 3 đơn hàng**?”
- “Phòng ban nào có **ít nhất 5 nhân viên**?”
- “Sản phẩm nào được **mua nhiều hơn một lần**?”

Công thức tư duy: **xác định đối tượng nhóm → chọn cột `GROUP BY` → xác định đại lượng tổng hợp → `HAVING` lọc nhóm → `SELECT` xuất đúng cột**.

Đừng nhầm với `WHERE`: nếu đề hỏi “những nhân viên lương > 5000” thì đó là lọc **từng dòng**, không phải đếm nhóm.

## 30. Biến thể 1: trả cả email và số lần xuất hiện

Nếu một bài khác yêu cầu xem **tần suất**, ta thêm `COUNT(*) AS occurrences` vào phần `SELECT`. **Đây là biến thể tự học, KHÔNG phải output của #182.**

```sql
SELECT email, COUNT(*) AS occurrences
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
```

Ta đang sử dụng đúng nhóm và điều kiện, chỉ thay đổi **đầu ra**. Học cách phân biệt “tư duy lõi” và “format output” sẽ giúp làm được nhiều bài SQL hơn.

In [24]:
show_182('SELECT email, COUNT(*) AS occurrences FROM Person GROUP BY email HAVING COUNT(*) > 1;')

Cột: ['email', 'occurrences']
   ('a@b.com', 2)
Tổng số dòng: 1


[('a@b.com', 2)]

## 31. Biến thể 2: tìm email xuất hiện ít nhất `k` lần

Đề gốc cố định ngưỡng **2**. Nếu muốn tìm email xuất hiện **ít nhất k lần**, ta thay bằng `HAVING COUNT(*) >= k`.

Ví dụ `k = 3` và các email `a, a, a, b, b` → chỉ chọn `a`. Khi chạy code Python gửi tham số SQL, dùng `?` và truyền `(k,)` thay vì nối trực tiếp chuỗi người dùng vào SQL.

**Lưu ý:** biểu thức `k` ở đây là biến Python dùng cho phòng lab, không phải tên cột tự có trong bảng Person.

In [25]:
def duplicate_at_least_k(rows, k):
    db = sqlite3.connect(':memory:')
    try:
        db.execute('CREATE TABLE Person (id INTEGER PRIMARY KEY, email TEXT NOT NULL)')
        db.executemany('INSERT INTO Person VALUES (?,?)', rows)
        return db.execute('SELECT email FROM Person GROUP BY email HAVING COUNT(*) >= ?', (k,)).fetchall()
    finally:
        db.close()

sample_k = list(enumerate(['a','a','a','b','b','c'], start=1))
print('k=2:', duplicate_at_least_k(sample_k, 2))
print('k=3:', duplicate_at_least_k(sample_k, 3))

k=2: [('a',), ('b',)]
k=3: [('a',)]


## 32. Biến thể 3: lọc *dòng* trước rồi mới nhóm

Hãy tưởng tượng một bảng email có nhiều kiểu dữ liệu, và ta chỉ muốn đếm **những email có đuôi `@b.com`**:

```sql
SELECT email, COUNT(*)
FROM Person
WHERE email LIKE '%@b.com'
GROUP BY email
HAVING COUNT(*) > 1;
```

`LIKE` là kiểm tra mẫu chuỗi; ký hiệu `%` đại diện cho **0 hoặc nhiều ký tự**. `WHERE` lọc **dòng phù hợp** trước, rồi `GROUP BY` chỉ nhận các dòng còn lại, và `HAVING` lọc **nhóm**.

Đây là mở rộng ngoài yêu cầu #182, nhằm làm rõ vì sao **có thể dùng WHERE và HAVING trong cùng một truy vấn**, miễn đặt đúng vai trò.

In [26]:
show_182("SELECT email, COUNT(*) AS so_lan FROM Person WHERE email LIKE '%@b.com' GROUP BY email HAVING COUNT(*) > 1;")

Cột: ['email', 'so_lan']
   ('a@b.com', 2)
Tổng số dòng: 1


[('a@b.com', 2)]

# 33. TỰ LUYỆN — 6 bài từ dễ đến khó

**Quy tắc luyện:** thử bằng tay trước → sau đó viết SQL/Python → chỉ đọc gợi ý nếu bí → đáp án đặt sau **toàn bộ bài tập**. Các cell TODO dưới đây **chỉ định nghĩa biến/hàm**, không tự thực thi code chưa hoàn thiện. Vì vậy `Run All` vẫn chạy an toàn.

Bạn đã có đủ công cụ: `SELECT`, `GROUP BY`, `COUNT`, `HAVING`, `AS`, `DISTINCT`, `JOIN` và Python dict. Bây giờ mục tiêu là tự **làm lại được**, không chỉ đọc thấy hợp lý.

### Bài tập 1 — Dự đoán output (`WHERE` và `DISTINCT`)

Cho email trong bảng lần lượt là `a, a, b, c, c`.

1. `SELECT DISTINCT email` sẽ trả các email nào?
2. `GROUP BY email HAVING COUNT(*) > 1` trả email nào?
3. Nếu thay bằng `HAVING COUNT(*) > 2` thì sao?
4. Câu nộp bắt buộc có bao nhiêu cột? Tên cột là gì?

**Tự ghi câu trả lời ra giấy trước.**

In [27]:
# Bài 1: tự dự đoán, sau đó BẠN có thể bỏ dấu # để kiểm tra.
# columns, rows = run_182([(1,'a'),(2,'a'),(3,'b'),(4,'c'),(5,'c')])
# print(columns, rows)
pass

### Bài tập 2 — Trace thủ công theo từng nhóm

Với bảng email `x, y, x, z, y, x, t`, hãy điền:

| email | số dòng | `COUNT(*) > 1` | giữ/bỏ |
|---|---:|---|---|
| x | ? | ? | ? |
| y | ? | ? | ? |
| z | ? | ? | ? |
| t | ? | ? | ? |

**Thử nói to:** “Tôi đang đếm **các dòng trong mỗi nhóm**, không phải đếm số nhóm”.

In [28]:
# Bài 2: tự tính bằng tay; cell trống để ghi kết quả.
trace_answer_182 = None  # TODO: tự thay bằng dict {'x':3, ...} khi đã làm xong

### Bài tập 3 — Điền TODO câu SQL đúng đề gốc

Tự hoàn thành câu SQL với đúng **một cột `Email`**. Bạn có thể lưu thành chuỗi Python để gọi `run_182`, nhưng đây chỉ là cách thử, khi nộp LeetCode bạn dán câu SQL.

**Không xem lại mục 15 ngay:** tự trả lời từ “nhóm theo gì?”, “đếm gì?”, “điều kiện bao nhiêu?” trước.

In [29]:
exercise3_sql182 = None
# TODO 1: SELECT email AS Email
# TODO 2: FROM Person
# TODO 3: GROUP BY email
# TODO 4: HAVING COUNT(*) > 1
# Sau khi tự làm: exercise3_sql182 = """..."""
# Khi đã hoàn thành, bạn có thể test:
# assert run_182([(1,'a'),(2,'b'),(3,'a')], exercise3_sql182)[1] == [('a',)]

### Bài tập 4 — Viết hàm Python đếm tần suất bằng dictionary

Hàm nhận `rows = [(id, email), ...]`, trả **danh sách các email trùng** đã sắp xếp.

Phải có: `counts = {}`, một vòng `for` đếm tần suất, một vòng lọc `count > 1`. Đây là cách luyện `dict`, không phải cách submit SQL.

In [30]:
def exercise4_counts182(rows):
    # TODO 1: tạo dictionary counts
    # TODO 2: duyệt (_, email) trong rows; tăng count
    # TODO 3: lọc email xuất hiện từ 2 lần trở lên
    # TODO 4: return danh sách sorted(...) để so sánh dễ dàng
    pass

# Sau khi viết xong, tự bật kiểm thử:
# assert exercise4_counts182([(1,'a'),(2,'a'),(3,'b')]) == ['a']

### Bài tập 5 — Viết lời giải `SELF JOIN` thay vì `GROUP BY`

Dùng `Person p1` và `Person p2`. Mục tiêu là chỉ chọn email có **ít nhất hai dòng với id khác nhau**.

Bạn cần tự điền: `JOIN`, điều kiện `email` giống nhau, điều kiện `id` khác nhau, và `DISTINCT` để tránh lặp kết quả. Thử phản ví dụ **bảng chỉ có một người** để phát hiện lỗi tự ghép chính nó.

In [31]:
exercise5_sql182 = None
# TODO: viết truy vấn SELECT DISTINCT ... AS Email FROM Person p1 JOIN Person p2 ON ...
# Sau đó bật:
# assert set(run_182([(1,'a'),(2,'b'),(3,'a')], exercise5_sql182)[1]) == {('a',)}

### Bài tập 6 — Mini challenge: email lặp ít nhất `k` lần

Viết hàm `exercise6_at_least_k182(rows, k)` dùng SQL để trả danh sách email có **tần suất >= k**.

- Với `a,a,a,b,b,c`, k=2 → `['a', 'b']`.
- k=3 → `['a']`.
- k=4 → `[]`.

**Thách thức:** dùng dấu `?` và tham số `(k,)` thay vì ghép chuỗi `str(k)` vào câu SQL; nhớ đóng database. Đây là bài mở rộng, **không** phải yêu cầu đề #182.

In [32]:
def exercise6_at_least_k182(rows, k):
    # TODO 1: tạo sqlite3.connect(':memory:')
    # TODO 2: tạo Person, chèn rows
    # TODO 3: GROUP BY email HAVING COUNT(*) >= ?
    # TODO 4: lấy cột email, sorted và đóng kết nối
    pass

# Sau khi viết, hãy tự test k=1,2,3,4; bảng rỗng.

## 34. GỢI Ý 3 CẤP cho cả 6 bài — đọc từ cấp nhẹ tới cấp gần-code

| Bài | Cấp 1 — nhắc ý | Cấp 2 — công cụ | Cấp 3 — gần-code |
|---|---|---|---|
| **1** | DISTINCT khác lọc theo số lần | Phân biệt “giá trị duy nhất” và “xuất hiện >=2” | DISTINCT: a,b,c; HAVING >1: a,c; HAVING >2: rỗng |
| **2** | Đếm số chữ trong danh sách | `x` có ba lần, `y` hai | Chỉ x và y có COUNT > 1 |
| **3** | Gom nhóm email rồi lọc nhóm | `GROUP BY email`; `HAVING COUNT(*)...` | `SELECT email AS Email ... HAVING COUNT(*) > 1` |
| **4** | Dùng dict lưu email → tần suất | `.get(email, 0)` | `counts[email] = counts.get(email,0)+1` |
| **5** | Tìm hai dòng có cùng email | `p1.email=p2.email` và `p1.id<>p2.id` | `SELECT DISTINCT p1.email AS Email ... JOIN ... ON ...` |
| **6** | Giống bài gốc nhưng ngưỡng biến đổi | `COUNT(*) >= ?` | `db.execute(query,(k,)).fetchall()` |

**Hướng giải quyết chung:** xác định **đầu ra** → mô phỏng bằng tay → chọn nhóm → chọn điều kiện → viết SQL ngắn → kiểm thử ca biên.

# 35. ĐÁP ÁN THAM KHẢO — Chỉ đọc sau khi tự luyện

## Đáp án bài 1 và 2 (có giải thích)

**Bài 1:**
- `DISTINCT email` → a, b, c (loại dòng lặp, **không lọc nhóm**).
- `HAVING COUNT(*) > 1` → a, c; vì mỗi email này xuất hiện 2 lần.
- `HAVING COUNT(*) > 2` → **rỗng**; không email nào xuất hiện 3 lần.
- Output gốc có **1 cột tên `Email`**.

**Bài 2:**

| email | số dòng | > 1? | Kết luận |
|---|---:|---|---|
| x | 3 | Đúng | giữ |
| y | 2 | Đúng | giữ |
| z | 1 | Sai | bỏ |
| t | 1 | Sai | bỏ |

Đếm thực sự bao nhiêu dòng trong **mỗi nhóm** là kỹ thuật trung tâm của bài.

In [33]:
print('Kiểm tra đáp án bài 1:')
_, rows = run_182(list(enumerate(['a','a','b','c','c'], start=1)))
print('SQL trả:', sorted(x[0] for x in rows))
assert sorted(x[0] for x in rows) == ['a','c']
print('Bài 2: x=3, y=2, z=1, t=1 → x và y')

Kiểm tra đáp án bài 1:
SQL trả: ['a', 'c']
Bài 2: x=3, y=2, z=1, t=1 → x và y


### Đáp án bài 3 — Truy vấn SQL chuẩn

Không cần thêm `DISTINCT`: nhóm theo email đã đảm bảo mỗi nhóm tạo một dòng. `HAVING COUNT(*) > 1` vừa đủ để lấy tất cả email bị trùng. Dòng `SELECT email AS Email` chỉ phục vụ **đúng format đầu ra**.

In [34]:
answer3_sql182 = '''SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;'''
print('Bài 3:',run_182([(1,'a'),(2,'b'),(3,'a')],answer3_sql182))
assert run_182([(1,'a'),(2,'b'),(3,'a')],answer3_sql182)[1] == [('a',)]

Bài 3: (['Email'], [('a',)])


### Đáp án bài 4 — Dictionary lưu số lần xuất hiện

Mỗi lần thấy email `e`, ta lấy số đếm trước đó (`counts.get(e, 0)`) rồi cộng 1. Lọc `count > 1` sau khi **đã đọc hết các dòng**. Không thêm email vào output mỗi lần bắt gặp vì phải **tránh trùng trong kết quả**.

In [35]:
def answer4_counts182(rows):
    counts = {}
    for _, email in rows:
        counts[email] = counts.get(email, 0) + 1
    return sorted(email for email, count in counts.items() if count > 1)

assert answer4_counts182([(1,'a'),(2,'b'),(3,'a')]) == ['a']
assert answer4_counts182([]) == []
print('PASS bài 4')

PASS bài 4


### Đáp án bài 5 — SELF JOIN và `DISTINCT`

`p1.id <> p2.id` xác định **hai bản ghi khác nhau**; `p1.email = p2.email` xác định cùng email. Nếu email xuất hiện nhiều lần, có nhiều cặp `p1,p2`, nên `DISTINCT` giữ đúng một email ở đầu ra.

In [36]:
answer5_sql182 = '''SELECT DISTINCT p1.email AS Email
FROM Person p1
JOIN Person p2 ON p1.email = p2.email AND p1.id <> p2.id;'''
for name, rows, expected in cases182:
    columns, actual = run_182(rows, answer5_sql182)
    assert columns == ['Email'] and {r[0] for r in actual} == expected
print('PASS bài 5 trên tất cả edge cases')

PASS bài 5 trên tất cả edge cases


### Đáp án bài 6 — Ngưỡng `k` có tham số

Thay điều kiện cố định bằng `HAVING COUNT(*) >= ?`. Dấu hỏi lấy giá trị từ `(k,)`: cách thực thi SQL có tham số. `try/finally` giúp đóng database cả khi lỗi. Sắp xếp **sau khi lấy dữ liệu** để việc test cho ra thứ tự cố định.

In [37]:
def answer6_at_least_k182(rows, k):
    db = sqlite3.connect(':memory:')
    try:
        db.execute('CREATE TABLE Person (id INTEGER PRIMARY KEY, email TEXT NOT NULL)')
        db.executemany('INSERT INTO Person VALUES (?, ?)', rows)
        found = db.execute('SELECT email FROM Person GROUP BY email HAVING COUNT(*) >= ?', (k,)).fetchall()
        return sorted(email for (email,) in found)
    finally:
        db.close()

rows = list(enumerate(['a','a','a','b','b','c'], start=1))
for k, expected in [(1,['a','b','c']),(2,['a','b']),(3,['a']),(4,[])]:
    assert answer6_at_least_k182(rows, k) == expected
print('PASS bài 6: k=1..4')

PASS bài 6: k=1..4


## 36. Template tư duy nhanh & checklist trước khi nộp

**Khi gặp đề kiểu “giá trị nào xuất hiện lặp?”**

1. Cần nhóm theo cột nào? → `GROUP BY email`.
2. Đếm cái gì? → `COUNT(*)`.
3. Chọn nhóm theo ngưỡng nào? → `HAVING COUNT(*) > 1`.
4. Xuất những cột nào? → `SELECT email AS Email`.

**SQL nộp #182:**

```sql
SELECT email AS Email
FROM Person
GROUP BY email
HAVING COUNT(*) > 1;
```

**Tự kiểm:** [ ] email đúng tên cột [ ] mỗi email chỉ một dòng [ ] nhóm đếm đúng 2 vẫn được lấy [ ] `HAVING` thay vì `WHERE COUNT` [ ] không trả `id`/`count` thừa [ ] không dán Python.

**Kết nối bài khác:** kỹ thuật `GROUP BY/HAVING` dùng rộng rãi trong phân tích dữ liệu (khách hàng mua nhiều lần, sản phẩm bán chạy, nhật ký bị trùng...). Học cách tự phân tích **đối tượng nhóm** quan trọng hơn nhớ một câu SQL cụ thể.